# Predicción de Clientes Próximos a Comprar

Calcula un score dinámico de recompra combinando frecuencia de compra, recencia y valor histórico del cliente, para priorizar campañas comerciales por impacto económico esperado.

**Fixes aplicados respecto a la versión original:**
- `variacion_gasto` ahora se protege contra división por cero y se acota a `[0, 1]` (antes podía superar 1 y volver negativa la probabilidad de recompra).
- `frecuencia_promedio` se acota con mínimo de 1 día, evitando división por cero en `score_recompra`.
- Se reporta cuántos clientes quedan excluidos por tener una sola compra (no se puede calcular frecuencia).
- Se eliminó el bloque de código muerto (lógica vieja comentada dentro de un docstring).
- Validación explícita de columnas requeridas.
- Nombres de cliente normalizados (`strip` + `upper`).
- Umbrales de filtrado (`0.35`, percentil 50) como constantes nombradas al inicio.
- El montaje de Google Drive solo se ejecuta si el notebook corre en Colab.


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------
ARCHIVO = r"/content/drive/MyDrive/Sales BILANNATURALS 2023-Mar2026.xlsm.xlsx"
HOJA = "Reporte Ventas"
COLUMNAS_REQUERIDAS = ["fecha", "cliente", "total", "familia", "cantidad"]

UMBRAL_PROB_MINIMA = 0.35
PERCENTIL_SCORE_MINIMO = 0.50
AJUSTE_DIAS_PROX_COMPRA = 3

# Intentar montar Google Drive solo si estamos en Colab
try:
    from google.colab import drive, files
    drive.mount('/content/drive', force_remount=False)
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
    print("Ejecutando fuera de Google Colab: se omite el montaje de Drive.")


In [ ]:
# ---------------------------------------------------------
# 1. CARGA Y LIMPIEZA DE DATOS
# ---------------------------------------------------------

def cargar_datos(archivo, hoja, columnas_requeridas):
    """Carga el Excel de ventas, valida columnas y limpia tipos de dato."""
    df = pd.read_excel(archivo, sheet_name=hoja)
    df.columns = df.columns.str.lower().str.strip()

    faltantes = [c for c in columnas_requeridas if c not in df.columns]
    if faltantes:
        raise ValueError(
            f"Faltan columnas requeridas en el Excel: {faltantes}. "
            f"Columnas encontradas: {list(df.columns)}"
        )

    df["fecha"] = pd.to_datetime(df["fecha"], dayfirst=True, errors="coerce")
    df["total"] = pd.to_numeric(df["total"], errors="coerce")
    df["cliente"] = df["cliente"].astype(str).str.strip().str.upper()

    filas_antes = len(df)
    df = df.dropna(subset=["fecha", "cliente", "total"])
    filas_descartadas = filas_antes - len(df)
    if filas_descartadas > 0:
        print(f"Se descartaron {filas_descartadas} filas por datos inválidos.")

    return df


In [ ]:
# ---------------------------------------------------------
# 2. CONSOLIDAR COMPRAS REALES (cliente + fecha = una compra)
# ---------------------------------------------------------

def consolidar_compras(df):
    df_compra = (
        df.groupby(["cliente", "fecha"])
        .agg(
            total_compra=("total", "sum"),
            familias=("familia", "nunique"),
            items=("cantidad", "sum"),
        )
        .reset_index()
        .sort_values(["cliente", "fecha"])
    )
    return df_compra


In [ ]:
# ---------------------------------------------------------
# 3. FRECUENCIA REAL DE COMPRA
# ---------------------------------------------------------

def calcular_frecuencia(df_compra):
    df_compra = df_compra.copy()
    df_compra["delta_dias"] = df_compra.groupby("cliente")["fecha"].diff().dt.days

    frecuencia = (
        df_compra.groupby("cliente")["delta_dias"]
        .agg(["mean", "std"])
        .reset_index()
        .rename(columns={"mean": "frecuencia_promedio", "std": "frecuencia_std"})
    )

    total_clientes = frecuencia["cliente"].nunique()
    frecuencia = frecuencia.dropna(subset=["frecuencia_promedio"])
    clientes_excluidos = total_clientes - frecuencia["cliente"].nunique()
    if clientes_excluidos > 0:
        print(
            f"Se excluyeron {clientes_excluidos} clientes con una sola compra "
            "(no se puede calcular frecuencia)."
        )

    frecuencia["frecuencia_promedio"] = frecuencia["frecuencia_promedio"].round().astype(int)
    # Evitar división por cero más adelante: frecuencia mínima de 1 día
    frecuencia["frecuencia_promedio"] = frecuencia["frecuencia_promedio"].clip(lower=1)
    frecuencia["frecuencia_std"] = frecuencia["frecuencia_std"].fillna(0)

    # ---------------------------------------------------------
    # 4. CLASIFICAR FRECUENCIA POR RANGOS
    # ---------------------------------------------------------
    bins = [0, 30, 60, 90, 120, 150, 10000]
    labels = ["1-30 dias", "31-60 dias", "61-90 dias", "91-120 dias", "121-150 dias", "150+ dias"]
    frecuencia["rango_frecuencia"] = pd.cut(
        frecuencia["frecuencia_promedio"], bins=bins, labels=labels, right=True
    )
    return frecuencia


In [ ]:
# ---------------------------------------------------------
# 5.1 FEATURES DE VALOR POR CLIENTE + FAMILIA MÁS COMPRADA
# ---------------------------------------------------------

def calcular_valor_por_cliente(df_compra, df_original):
    df_valor = (
        df_compra.groupby("cliente")
        .agg(
            total_promedio=("total_compra", "mean"),
            total_std=("total_compra", "std"),
            cantidad_promedio=("items", "mean"),
            num_familias=("familias", "nunique"),
        )
        .reset_index()
    )
    df_valor["total_std"] = df_valor["total_std"].fillna(0)

    familia_top = (
        df_original.groupby(["cliente", "familia"]).size().reset_index(name="conteo")
    )
    familia_top = (
        familia_top.sort_values(["cliente", "conteo"], ascending=False)
        .drop_duplicates("cliente")
    )

    df_valor = df_valor.merge(familia_top[["cliente", "familia"]], on="cliente", how="left")
    return df_valor


In [ ]:
# ---------------------------------------------------------
# 5. SCORE DE RECOMPRA, PROBABILIDAD Y VALOR ESPERADO
# ---------------------------------------------------------

def calcular_scores(df_compra, frecuencia, df_valor):
    ultima_compra = (
        df_compra.groupby("cliente")["fecha"]
        .max()
        .reset_index()
        .rename(columns={"fecha": "ultima_fecha_compra"})
    )

    resultado = ultima_compra.merge(frecuencia, on="cliente", how="inner")
    resultado = resultado.merge(df_valor, on="cliente", how="left")

    fecha_hoy = pd.Timestamp(datetime.now().date())
    resultado["dias_desde_ultima"] = (fecha_hoy - resultado["ultima_fecha_compra"]).dt.days

    # Próxima compra estimada
    resultado["prox_compra_calculada"] = (
        resultado["ultima_fecha_compra"]
        - pd.Timedelta(days=AJUSTE_DIAS_PROX_COMPRA)
        + pd.to_timedelta(resultado["frecuencia_promedio"], unit="D")
    )

    # SCORE DINÁMICO DE RECOMPRA (frecuencia_promedio ya está protegida con clip(lower=1))
    resultado["score_recompra"] = resultado["dias_desde_ultima"] / resultado["frecuencia_promedio"]

    # Ventana probabilística (descriptiva; no se usa en el filtro final, se mantiene para análisis exploratorio)
    resultado["limite_inferior"] = resultado["frecuencia_promedio"] - resultado["frecuencia_std"]
    resultado["limite_superior"] = resultado["frecuencia_promedio"] + resultado["frecuencia_std"]

    # VARIACIÓN DE GASTO: protegida contra división por cero y acotada a [0, 1]
    # (antes podía superar 1 y volver negativa la probabilidad de recompra)
    resultado["variacion_gasto"] = np.where(
        resultado["total_promedio"] > 0,
        resultado["total_std"] / resultado["total_promedio"],
        0,
    )
    resultado["variacion_gasto"] = resultado["variacion_gasto"].clip(0, 1)

    # PROBABILIDAD DE RECOMPRA: penaliza inestabilidad de gasto, acotada [0, 1]
    resultado["prob_recompra"] = resultado["score_recompra"] * (1 - resultado["variacion_gasto"])
    resultado["prob_recompra"] = resultado["prob_recompra"].clip(0, 1)

    # VALOR ESPERADO DE RECOMPRA
    resultado["valor_estimado"] = resultado["total_promedio"] * (1 + resultado["variacion_gasto"])

    # SCORE FINAL: combina probabilidad y valor económico esperado
    resultado["score_final"] = resultado["prob_recompra"] * resultado["valor_estimado"]

    return resultado


In [ ]:
# ---------------------------------------------------------
# 6. FILTRO FINAL INTELIGENTE
# ---------------------------------------------------------

def filtrar_clientes_prioritarios(resultado, umbral_prob, percentil_score):
    umbral_score = resultado["score_final"].quantile(percentil_score)
    filtrado = resultado[
        (resultado["prob_recompra"] >= umbral_prob) &
        (resultado["score_final"] > umbral_score)
    ].copy()
    filtrado = filtrado.sort_values("score_final", ascending=False)
    return filtrado


In [ ]:
# ---------------------------------------------------------
# EJECUCIÓN
# ---------------------------------------------------------
df = cargar_datos(ARCHIVO, HOJA, COLUMNAS_REQUERIDAS)
df_compra = consolidar_compras(df)
frecuencia = calcular_frecuencia(df_compra)
df_valor = calcular_valor_por_cliente(df_compra, df)
resultado_final = calcular_scores(df_compra, frecuencia, df_valor)
resultado_filtrado = filtrar_clientes_prioritarios(
    resultado_final, UMBRAL_PROB_MINIMA, PERCENTIL_SCORE_MINIMO
)

print(f"Clientes filtrados: {resultado_filtrado.shape[0]}")
print(
    f"Filtro: prob_recompra >= {UMBRAL_PROB_MINIMA} "
    f"y score_final > percentil {int(PERCENTIL_SCORE_MINIMO * 100)}"
)

# ---------------------------------------------------------
# 7. EXPORTAR EXCEL FINAL
# ---------------------------------------------------------
fecha_hoy_str = datetime.now().strftime("%d-%m-%Y")
nombre_excel = f"CLIENTES PROX A COMPRAR {fecha_hoy_str}.xlsx"

with pd.ExcelWriter(nombre_excel) as writer:
    resultado_filtrado.to_excel(writer, index=False)

print(f"Archivo generado correctamente: {nombre_excel}")
print(f"Total clientes exportados: {resultado_filtrado.shape[0]}")

if EN_COLAB:
    files.download(nombre_excel)
